<a href="https://colab.research.google.com/github/febriyansyah-id/COLLAB/blob/main/notebooks/preprocessing/preprocessing_superstore_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Penelitian Preprocessing Data dan Segmentasi Pelanggan Superstore

**Jenis studi:** eksperimen kuantitatif berbasis data transaksi  
**Dataset:** `Superstore_Modifikasi.xlsx`, diunggah pengguna saat notebook berjalan di Google Colab  
**Unit analisis:** baris transaksi untuk audit; pelanggan untuk clustering  
**Luaran:** data transaksi tervalidasi, fitur RFM dan cluster pelanggan, serta audit proses

## Abstrak

Penelitian ini mengkaji pengaruh penyiapan data transaksi terhadap pembentukan segmen pelanggan. Dataset Superstore yang dimodifikasi diaudit untuk menemukan nilai kosong, baris duplikat, ketidaksesuaian tipe, dan angka yang tidak valid. Data yang dapat dipastikan dipulihkan dengan aturan satu-ke-satu, sedangkan nilai finansial yang tidak diketahui tetap kosong. Transaksi yang memenuhi syarat digunakan untuk menghitung *recency*, *frequency*, dan *monetary* (RFM) per pelanggan. Fitur RFM ditransformasi dan dikelompokkan dengan K-Means; jumlah cluster dipilih menggunakan skor silhouette. Notebook menghasilkan audit kuantitatif, profil cluster, serta kesimpulan otomatis dari data yang diunggah. Hasil numerik ditampilkan saat seluruh cell dijalankan, sehingga abstrak ini tidak mengasumsikan jumlah baris atau cluster tertentu.

**Kata kunci:** kualitas data, preprocessing, RFM, segmentasi pelanggan, K-Means, silhouette.

**Cara menjalankan:** jalankan cell dari atas ke bawah; saat pemilih berkas muncul, unggah satu file `.xlsx`. Notebook tidak menggunakan path lokal komputer.

# BAB 1 — PENDAHULUAN

## 1.1 Latar Belakang

Data transaksi retail memuat jejak pembelian yang dapat diringkas menjadi perilaku pelanggan. RFM menggunakan kapan pelanggan terakhir membeli, seberapa sering ia memesan, dan berapa besar nilai pembeliannya; ketiga dimensi ini telah digunakan dalam kajian nilai pelanggan [1]. Studi retail lain menerapkan RFM bersama K-Means untuk mengidentifikasi kelompok pelanggan dengan pola pembelian yang berbeda [2], [6].

Nilai kosong, duplikat, dan angka yang tidak valid dapat mengubah jumlah transaksi, total penjualan, dan jarak antar pelanggan. Metode imputasi tersedia untuk data campuran [5], tetapi imputasi nilai penjualan tanpa dasar yang cukup dapat menambahkan nilai buatan ke fitur monetary. Karena itu, penelitian ini mencatat kehilangan data secara terbuka, memulihkan hanya informasi yang memiliki pasangan unik, dan membatasi perhitungan RFM pada transaksi yang memiliki kunci serta `Sales` valid. Keputusan tersebut merupakan pilihan metodologis penelitian ini, bukan klaim bahwa semua metode imputasi keliru.

K-Means mengelompokkan pelanggan berdasarkan kemiripan fitur, tetapi hasilnya bergantung pada representasi dan skala fitur [3]. Skor silhouette memberi ukuran internal untuk membandingkan pemisahan antar cluster [4]. Evaluasi kuantitatif perlu diikuti pembacaan profil RFM setiap cluster agar nomor cluster mendapat makna bisnis.

## 1.2 Rumusan Masalah

1. Seberapa besar masalah kualitas data pada file transaksi yang diunggah?
2. Berapa banyak baris yang berubah atau terbuang setelah preprocessing, dan mengapa?
3. Bagaimana pelanggan terkelompok berdasarkan RFM, berapa jumlah cluster yang dipilih, dan bagaimana profil tiap cluster?
4. Apa batasan interpretasi segmentasi ketika sebagian transaksi tidak lengkap?

## 1.3 Tujuan Penelitian

1. Mengukur nilai kosong, duplikat, nilai tidak valid, dan nilai ekstrem pada data sumber.
2. Menghasilkan data transaksi tervalidasi serta menjelaskan aturan pembersihannya.
3. Membentuk fitur RFM pelanggan dan melakukan clustering yang dapat diulang.
4. Menilai hasil dengan skor silhouette dan menafsirkan profil kelompok menggunakan median RFM.

## 1.4 Manfaat dan Batasan

Hasil dapat digunakan sebagai contoh alur preprocessing dan eksplorasi segmentasi retail. Cluster merupakan deskripsi pola transaksi pada data yang diamati; notebook ini tidak mengukur dampak strategi pemasaran, memprediksi nilai pelanggan masa depan, atau membuktikan hubungan sebab-akibat.

# BAB 2 — TINJAUAN PUSTAKA

## 2.1 Data transaksi retail dan unit analisis

Data Superstore berbentuk baris item pesanan. Satu `Order ID` dapat muncul pada beberapa baris karena sebuah pesanan berisi beberapa produk. Dengan demikian, **baris transaksi**, **pesanan**, dan **pelanggan** adalah unit yang berbeda. Audit kualitas dilakukan pada baris, *frequency* dihitung sebagai jumlah pesanan unik per pelanggan, sedangkan segmentasi dilakukan pada pelanggan. Kesalahan membedakan unit dapat memperbesar frequency pelanggan yang membeli banyak item dalam satu pesanan [1], [2]. Data yang dipakai penelitian ini berasal dari sheet `Data_Modifikasi` pada workbook yang diunggah pengguna, sehingga seluruh hasil empiris bergantung pada isi file tersebut.

## 2.2 Kualitas data sebagai prasyarat analisis

Kualitas data mencakup lebih dari akurasi; kelengkapan, konsistensi, dan kesesuaian data dengan kebutuhan pengguna juga menentukan kegunaannya [7]. Untuk penelitian ini, kelengkapan dinilai melalui jumlah sel kosong dan kelayakan kolom wajib; konsistensi dinilai melalui tipe, rentang nilai, serta hubungan antaridentitas dan tanggal. `Customer ID` yang kosong menghalangi pengelompokan pelanggan, `Order Date` yang kosong menghalangi recency, dan `Sales` yang kosong menurunkan monetary teramati. Karena pengaruhnya berbeda, jumlah kosong dilaporkan per kolom dan dibandingkan sebelum serta sesudah pembersihan, bukan hanya sebagai satu angka total.

Duplikat persis dipisahkan dari `Order ID` berulang. Duplikat persis ialah seluruh nilai baris sama, sehingga dihapus untuk mencegah penghitungan ganda. `Order ID` berulang masih dapat mewakili item berbeda dalam pesanan yang sah. Validasi aturan bisnis menandai `Sales <= 0`, `Quantity <= 0` atau bukan bilangan bulat, dan `Discount` di luar interval 0–1 sebagai nilai tidak valid. `Profit` negatif tetap mungkin terjadi pada transaksi rugi. Aturan ini merupakan keputusan operasional penelitian yang tercatat pada audit, bukan klaim bahwa semua data retail mengikuti pola yang sama.

## 2.3 Nilai hilang dan batas pemulihan

Metode imputasi dapat mengisi nilai hilang, tetapi hasilnya bergantung pada asumsi model dan pola kehilangan data [5]. Analisis hanya pada observasi lengkap juga dapat mengubah komposisi sampel ketika kehilangan data berkaitan dengan karakteristik observasi [9]. Oleh sebab itu, penelitian ini memulihkan `Customer ID` atau `Order Date` **hanya ketika** pasangan identitas/pesanan memberikan satu nilai yang konsisten. Kategori kosong diberi label `Unknown`; atribut wajib RFM yang tetap kosong tidak ditebak. Baris tersebut tetap ada dalam data transaksi tervalidasi, tetapi tidak masuk basis RFM. Jumlah dan proporsinya ditampilkan agar pembaca dapat menilai cakupan segmentasi. Kebijakan ini menjaga keterlacakan, tetapi tidak menghapus kemungkinan bias seleksi pada pelanggan yang datanya tidak lengkap [9].

## 2.4 Pencilan dan sebaran transaksi

Aturan rentang antarkuartil (IQR) menggunakan batas bawah `Q1 − 1,5 × IQR` dan batas atas `Q3 + 1,5 × IQR`. Pada distribusi yang menceng, aturan boxplot biasa dapat menandai banyak observasi pada salah satu sisi; penelitian tentang *adjusted boxplot* menjelaskan masalah ini [10]. Nilai `Sales` atau `Profit` yang ekstrem dapat mewakili transaksi besar yang sah. Karena itu, notebook **melaporkan** jumlah pencilan pada `Sales`, `Quantity`, `Discount`, dan `Profit` tanpa menghapusnya otomatis. Untuk mengurangi dominasi nilai sangat tinggi pada jarak K-Means, pembatasan persentil 99 hanya dilakukan pada **salinan fitur pemodelan**. Nilai transaksi dan RFM asli tetap dipakai untuk tabel interpretasi.

## 2.5 Model Recency, Frequency, Monetary (RFM)

RFM adalah ringkasan perilaku pelanggan yang memakai kapan terakhir membeli, seberapa sering membeli, dan berapa nilai pembelian [1]. Kajian Fader dkk. menghubungkan informasi RFM dengan analisis nilai pelanggan, sedangkan Anitha dan Patil menerapkannya bersama K-Means untuk perilaku pembelian [1], [2]. Pada penelitian ini, untuk pelanggan ke-*i*:

- **Recency** `Rᵢ = tanggal_acuan − tanggal_pesanan_terakhirᵢ` dalam hari; lebih kecil berarti lebih baru.
- **Frequency** `Fᵢ = jumlah Order ID unikᵢ`; menghitung pesanan, bukan baris item.
- **Monetary** `Mᵢ = Σ Salesᵢ` pada baris yang valid dan teramati.

Tanggal acuan adalah satu hari setelah tanggal pesanan maksimum pada baris layak RFM. Dengan demikian, recency minimum tetap positif dan dapat dipakai dalam transformasi `log1p`. Monetary ialah nilai teramati, bukan estimasi pendapatan sejati jika sebagian transaksi tidak memiliki `Sales`.

## 2.6 K-Means dan pengolahan fitur

K-Means membagi pelanggan menjadi *k* kelompok dengan meminimalkan jumlah kuadrat jarak observasi terhadap pusat kelompoknya [3]. Tinjauan algoritma clustering menekankan bahwa pilihan ukuran jarak, representasi fitur, dan validasi memengaruhi hasil pengelompokan [8]. Karena RFM berbeda satuan dan sering memiliki rentang menceng, notebook menerapkan pembatasan persentil 99 pada salinan fitur, `log1p`, lalu standardisasi. Urutan ini membuat jarak lebih seimbang, tetapi juga dapat mengubah batas cluster; sebab itu profil kelompok dijelaskan memakai median RFM **asli**. `random_state=42` dan `n_init=10` ditetapkan agar hasil pada data dan versi pustaka yang sama dapat diulang. K-Means tetap mengasumsikan kelompok yang dapat dipisahkan lewat kedekatan pada ruang fitur; pola pelanggan yang kompleks belum tentu terwakili baik [8].

## 2.7 Validasi internal dengan silhouette

Untuk observasi *i*, `a(i)` ialah rerata jarak ke anggota cluster sendiri dan `b(i)` ialah rerata jarak terendah ke cluster lain. Skor `s(i) = [b(i) − a(i)] / max[a(i), b(i)]` berada antara −1 dan 1; nilai lebih tinggi menunjukkan pemisahan internal yang lebih baik [4]. Notebook membandingkan rata-rata silhouette untuk kandidat `k=2` hingga `k=8`, sepanjang jumlah pelanggan memungkinkan. Silhouette membantu memilih *k* pada data yang tersedia, tetapi tidak membuktikan stabilitas antarperiode, kegunaan bisnis, atau hubungan sebab-akibat. Jika fitur terlalu seragam, notebook melaporkan satu kelompok tanpa mengklaim kualitas pemisahan.

## 2.8 Penelitian terkait dan sintesis

| Rujukan | Fokus utama | Relevansi dan batas penerapan di studi ini |
|---|---|---|
| Fader dkk. [1] | Informasi RFM dalam analisis nilai pelanggan | Dasar tiga dimensi perilaku; notebook ini tidak mengestimasi *customer lifetime value*. |
| Anitha dan Patil [2] | RFM dengan K-Means untuk perilaku pembelian | Menunjukkan pendekatan segmentasi; kualitas data sumber tetap perlu diaudit terpisah. |
| Stekhoven dan Bühlmann [5] | Imputasi data campuran dengan MissForest | Menegaskan adanya opsi imputasi; penelitian ini memilih pemulihan berbasis aturan yang dapat diaudit. |
| Awaliyah dkk. [6] | K-Means pada transaksi retail daring | Pembanding penerapan segmentasi; dataset dan profil kelompok tidak dianggap sama. |
| Wang dan Strong [7] | Dimensi kualitas data dari sudut konsumen data | Landasan audit kelengkapan dan kesesuaian penggunaan. |
| Xu dan Wunsch [8] | Survei algoritma clustering dan validasi | Landasan keterbatasan pemilihan algoritma dan jarak. |
| Hubert dan Vandervieren [10] | Boxplot pada distribusi menceng | Alasan agar nilai ekstrem tidak langsung dibuang. |

## 2.9 Kerangka konseptual dan posisi penelitian

Alur penelitian adalah **unggah workbook → audit kualitas → validasi dan pemulihan yang dapat dipastikan → pembentukan RFM pelanggan → transformasi fitur → K-Means dan silhouette → profil cluster → analisis otomatis**. Hubungan ini bersifat prosedural: kualitas data menentukan baris yang layak, baris layak menentukan RFM, dan RFM menjadi masukan clustering. Kontribusi studi ini ialah menggabungkan pelaporan kualitas data dan segmentasi dalam satu notebook Colab dengan tabel, grafik, serta interpretasi yang dihitung saat dijalankan. Klaimnya terbatas pada pola deskriptif dataset yang diunggah; validitas bisnis memerlukan pemeriksaan domain dan data periode lain.

# BAB 3 — METODOLOGI PENELITIAN

## 3.1 Desain dan Sumber Data

Penelitian menggunakan desain eksperimen komputasional deskriptif. Pengguna mengunggah workbook `Superstore_Modifikasi.xlsx` saat notebook dijalankan. Sheet `Data_Modifikasi` menjadi data utama dan sheet `Ringkasan`, bila tersedia, ditampilkan sebagai informasi asal dataset. Jumlah baris dan tingkat kerusakan dibaca dari file aktual, bukan ditulis tetap dalam laporan.

## 3.2 Unit Analisis dan Variabel

| Tahap | Unit | Variabel utama | Kegunaan |
|---|---|---|---|
| Audit | Baris transaksi | Seluruh kolom, terutama ID, tanggal, `Sales`, `Quantity`, `Discount`, `Profit` | Mengukur masalah kualitas data |
| Pembersihan | Baris transaksi | ID, teks, tanggal, numerik | Memvalidasi tipe dan aturan bisnis |
| RFM | Pelanggan | `recency_days`, `frequency`, `monetary` | Representasi perilaku untuk clustering |
| Evaluasi | Pelanggan dan cluster | Label cluster, skor silhouette, median RFM | Menilai pemisahan dan profil kelompok |

## 3.3 Prosedur Preprocessing

1. Baca workbook yang diunggah dan periksa kolom wajib.
2. Rapikan teks, normalisasi tipe, dan hapus duplikat baris persis.
3. Pulihkan `Customer ID` dari nama pelanggan atau pesanan, serta `Order Date` dari pesanan, hanya jika pemetaan satu-ke-satu konsisten.
4. Tandai `Sales <= 0`, `Quantity <= 0` atau bukan bilangan bulat, dan `Discount` di luar 0–1 sebagai tidak valid. `Profit` negatif tetap sah karena transaksi dapat rugi.
5. Catat nilai ekstrem dengan aturan IQR; jangan hapus otomatis nilai yang mungkin merupakan transaksi besar yang sah.
6. Isi kategori yang kosong dengan `Unknown`; biarkan ID, tanggal, dan nilai finansial yang tidak dapat dipastikan sebagai kosong. Baris dengan `Customer ID`, `Order ID`, `Order Date`, dan `Sales` valid menjadi basis RFM.

## 3.4 Pembentukan RFM

Tanggal acuan ditetapkan satu hari setelah `Order Date` maksimum pada transaksi layak pakai. Untuk pelanggan *i*:

- `Recency_i = tanggal_acuan − tanggal_pesanan_terakhir_i` (hari).
- `Frequency_i = jumlah Order ID unik pelanggan i`.
- `Monetary_i = jumlah Sales valid yang teramati untuk pelanggan i`.

Monetary dapat lebih rendah dari nilai riil pelanggan jika beberapa baris penjualan hilang; keterbatasan ini dilaporkan pada BAB 5.

## 3.5 Model dan Evaluasi

Nilai RFM untuk model dibatasi pada persentil 99 per fitur, ditransformasi `log1p`, kemudian distandardisasi. K-Means mencoba `k=2` sampai paling banyak `k=8` dengan `random_state=42` dan `n_init=10`. Nilai `k` dengan skor silhouette rata-rata tertinggi dipilih [3], [4]. Tabel cluster merangkum jumlah pelanggan dan median RFM untuk interpretasi. Jika data tidak cukup beragam untuk membentuk lebih dari satu kelompok, notebook melaporkannya tanpa memaksakan segmentasi.

## 3.6 Reproduksibilitas dan Luaran

Notebook menyimpan kode tanpa output eksekusi. Seluruh angka pada BAB 4 dihitung ulang ketika file diunggah. Hasil CSV dan laporan audit dikemas dalam ZIP untuk diunduh dari Colab. File Excel sumber tidak ditulis ulang.

# BAB 4 — IMPLEMENTASI, HASIL, DAN PEMBAHASAN

Seluruh script upload, audit, preprocessing, clustering, visualisasi, ekspor, dan analisis hasil berada di bab ini. Setiap output dihitung saat notebook dijalankan; angka yang muncul perlu dibaca bersama catatan kehilangan data dan keterbatasan evaluasi internal [4], [5].

## 4.1 Siapkan library dan unggah Excel

Script ini memasang pustaka, meminta satu workbook `.xlsx`, memvalidasi sheet dan kolom wajib, lalu menampilkan contoh lima baris transaksi. Jika sheet `Ringkasan` tersedia, tabelnya ikut ditampilkan. Keluaran otomatis setelah tabel menjelaskan ukuran data dan keberadaan sheet ringkasan.


In [ ]:
%pip -q install openpyxl scikit-learn

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from google.colab import files
from IPython.display import display

uploaded = files.upload()
excel_names = [name for name in uploaded if name.lower().endswith('.xlsx')]
if len(excel_names) != 1:
    raise ValueError('Unggah tepat satu file .xlsx pada cell ini.')
source_file = Path(excel_names[0])
book = pd.ExcelFile(source_file, engine='openpyxl')
if 'Data_Modifikasi' not in book.sheet_names:
    raise ValueError(f'Sheet Data_Modifikasi tidak ditemukan. Sheet tersedia: {book.sheet_names}')
raw = pd.read_excel(book, sheet_name='Data_Modifikasi')
source_summary = pd.read_excel(book, sheet_name='Ringkasan') if 'Ringkasan' in book.sheet_names else None
required_columns = {'Order ID', 'Order Date', 'Customer ID', 'Customer Name', 'Sales', 'Quantity', 'Discount', 'Profit'}
missing_columns = required_columns - set(raw.columns)
if missing_columns:
    raise ValueError(f'Kolom wajib tidak ditemukan: {sorted(missing_columns)}')
print(f'File: {source_file.name} | sheet: {book.sheet_names} | data: {raw.shape[0]:,} baris x {raw.shape[1]} kolom')
display(raw.head())
if source_summary is not None:
    display(source_summary)

print(f'DESKRIPSI OTOMATIS: workbook memiliki {len(book.sheet_names)} sheet; sheet Data_Modifikasi berisi {len(raw):,} baris dan {raw.shape[1]} kolom. Tabel pertama adalah cuplikan lima baris untuk memeriksa struktur, bukan ringkasan seluruh data.')
print('Tabel Ringkasan dari workbook juga ditampilkan.' if source_summary is not None else 'Sheet Ringkasan tidak tersedia; analisis tetap memakai Data_Modifikasi.')


## 4.2 Audit data awal

Script menghitung jumlah baris, sel kosong, dan duplikat persis pada data sumber. Tabel menunjukkan jumlah nilai kosong per kolom; uraian otomatis mengidentifikasi kolom yang paling membutuhkan perhatian.


In [ ]:
initial_rows = len(raw)
initial_missing = int(raw.isna().sum().sum())
initial_duplicates = int(raw.duplicated().sum())
initial_missing_by_column = raw.isna().sum().astype(int)
print(f'Baris: {initial_rows:,}; sel kosong: {initial_missing:,}; baris duplikat persis: {initial_duplicates:,}')
display(initial_missing_by_column.rename_axis('kolom').reset_index(name='jumlah_kosong'))

top_missing = initial_missing_by_column.sort_values(ascending=False)
if initial_missing:
    print(f'DESKRIPSI OTOMATIS: {initial_missing:,} sel kosong tersebar pada {(initial_missing_by_column > 0).sum()} kolom. Kolom paling banyak kosong adalah {top_missing.index[0]} ({int(top_missing.iloc[0]):,} sel).')
else:
    print('DESKRIPSI OTOMATIS: tidak ada sel kosong pada data sumber.')
print(f'Duplikat persis setara dengan {100 * initial_duplicates / initial_rows:.1f}% baris sumber.' if initial_rows else 'Data sumber tidak memiliki baris.')


## 4.3 Bersihkan nilai dan pulihkan informasi yang dapat dipastikan

Script menormalkan teks, tanggal, dan angka; menghapus duplikat persis; memulihkan identitas atau tanggal melalui pasangan unik; serta menandai angka dan tanggal yang tidak valid. Ringkasan otomatis menjelaskan perubahan jumlah baris, pemulihan, dan masalah yang masih tersisa.


In [ ]:
cleaned = raw.copy()
text_columns = [
    'Order ID', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment',
    'Country', 'City', 'State', 'Region', 'Product ID', 'Category',
    'Sub-Category', 'Product Name'
]
for column in text_columns:
    if column in cleaned.columns:
        cleaned[column] = cleaned[column].astype('string').str.strip().replace('', pd.NA)
for column in ['Order Date', 'Ship Date']:
    if column in cleaned.columns:
        cleaned[column] = pd.to_datetime(cleaned[column], errors='coerce')
for column in ['Row ID', 'Postal Code', 'Sales', 'Quantity', 'Discount', 'Profit']:
    if column in cleaned.columns:
        cleaned[column] = pd.to_numeric(cleaned[column], errors='coerce')

before_dedup = len(cleaned)
cleaned = cleaned.drop_duplicates().reset_index(drop=True)
duplicate_removed = before_dedup - len(cleaned)

def fill_from_unique_key(frame, target, key):
    """Isi target hanya bila key selalu terhubung ke satu nilai target."""
    pairs = frame[[key, target]].dropna().drop_duplicates()
    unique_keys = pairs.groupby(key)[target].nunique()
    valid_keys = unique_keys[unique_keys == 1].index
    mapping = pairs[pairs[key].isin(valid_keys)].drop_duplicates(key).set_index(key)[target]
    missing_before = int(frame[target].isna().sum())
    frame[target] = frame[target].fillna(frame[key].map(mapping))
    return missing_before - int(frame[target].isna().sum())

customer_id_from_name = fill_from_unique_key(cleaned, 'Customer ID', 'Customer Name')
customer_id_from_order = fill_from_unique_key(cleaned, 'Customer ID', 'Order ID')
order_date_from_order = fill_from_unique_key(cleaned, 'Order Date', 'Order ID')

invalid_sales = int((cleaned['Sales'] <= 0).fillna(False).sum())
invalid_quantity = int(((cleaned['Quantity'] <= 0) | (cleaned['Quantity'] % 1 != 0)).fillna(False).sum())
invalid_discount = int(((cleaned['Discount'] < 0) | (cleaned['Discount'] > 1)).fillna(False).sum())
cleaned.loc[cleaned['Sales'] <= 0, 'Sales'] = np.nan
cleaned.loc[(cleaned['Quantity'] <= 0) | (cleaned['Quantity'] % 1 != 0), 'Quantity'] = np.nan
cleaned.loc[(cleaned['Discount'] < 0) | (cleaned['Discount'] > 1), 'Discount'] = np.nan

ship_before_order = 0
if 'Ship Date' in cleaned.columns:
    bad_ship = cleaned['Ship Date'].notna() & cleaned['Order Date'].notna() & (cleaned['Ship Date'] < cleaned['Order Date'])
    ship_before_order = int(bad_ship.sum())
    cleaned.loc[bad_ship, 'Ship Date'] = pd.NaT

category_columns = ['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Region', 'Category', 'Sub-Category', 'Product Name']
for column in category_columns:
    if column in cleaned.columns:
        cleaned[column] = cleaned[column].fillna('Unknown')

print(f'Duplikat dihapus: {duplicate_removed:,}; Customer ID dipulihkan: {customer_id_from_name + customer_id_from_order:,}; Order Date dipulihkan: {order_date_from_order:,}')
print(f'Sales tidak valid: {invalid_sales:,}; Quantity tidak valid: {invalid_quantity:,}; Discount tidak valid: {invalid_discount:,}; Ship Date sebelum Order Date: {ship_before_order:,}')
print(f'Data transaksi tervalidasi: {len(cleaned):,} baris; nilai kosong tersisa: {int(cleaned.isna().sum().sum()):,} sel')

print(f'DESKRIPSI OTOMATIS: jumlah baris turun dari {before_dedup:,} menjadi {len(cleaned):,} karena penghapusan {duplicate_removed:,} duplikat persis.')
print(f'Pemulihan berbasis pasangan unik mengisi {customer_id_from_name + customer_id_from_order:,} Customer ID dan {order_date_from_order:,} Order Date; nilai yang tidak pasti tetap kosong.')
print(f'Validasi menandai {invalid_sales + invalid_quantity + invalid_discount:,} kejadian angka tidak valid dan {ship_before_order:,} Ship Date sebelum Order Date. Sel kosong setelah validasi: {int(cleaned.isna().sum().sum()):,}.')


## 4.4 Tinjau nilai ekstrem

Metode IQR digunakan untuk **melaporkan** nilai ekstrem. Baris yang hanya ekstrem tidak dihapus otomatis karena sebagian transaksi besar atau rugi dapat sah. Clipping persentil 99 dilakukan nanti pada salinan fitur clustering.

Script memakai batas IQR untuk membuat tabel jumlah nilai ekstrem pada variabel numerik. Uraian otomatis menunjukkan kolom dengan temuan terbanyak dan menegaskan bahwa nilai ekstrem belum dibuang.


In [ ]:
outlier_rows = []
for column in ['Sales', 'Quantity', 'Discount', 'Profit']:
    values = cleaned[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((values < lower) | (values > upper)).sum())
    outlier_rows.append({'kolom': column, 'batas_bawah_IQR': lower, 'batas_atas_IQR': upper, 'jumlah_di_luar_IQR': count})
outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

total_outlier = int(outlier_report['jumlah_di_luar_IQR'].sum())
if total_outlier:
    worst = outlier_report.sort_values('jumlah_di_luar_IQR', ascending=False).iloc[0]
    print(f'DESKRIPSI OTOMATIS: tabel IQR mencatat {total_outlier:,} penandaan nilai ekstrem di seluruh kolom; paling banyak pada {worst["kolom"]} ({int(worst["jumlah_di_luar_IQR"]):,}). Satu baris dapat terhitung pada lebih dari satu kolom.')
else:
    print('DESKRIPSI OTOMATIS: aturan IQR tidak menandai nilai ekstrem pada kolom yang diperiksa.')
print('Nilai ekstrem hanya dilaporkan; tidak ada baris yang dihapus karena aturan IQR.')


## 4.5 Bentuk data pelanggan RFM

Baris dengan kunci atau `Sales` yang masih kosong tidak digunakan untuk menghitung RFM. `Order ID` yang berulang pada beberapa item hanya dihitung sekali pada frequency.

Script memilih baris dengan kolom wajib lengkap, lalu menghitung recency, jumlah pesanan unik, dan total Sales teramati per pelanggan. Tabel menampilkan lima pelanggan pertama; uraian otomatis menjelaskan cakupan data dan rentang nilai RFM.


In [ ]:
model_required = ['Customer ID', 'Order ID', 'Order Date', 'Sales']
model_base = cleaned.dropna(subset=model_required).copy()
if model_base.empty:
    raise ValueError('Tidak ada transaksi lengkap untuk RFM. Periksa kolom wajib pada file sumber.')
reference_date = model_base['Order Date'].max() + pd.Timedelta(days=1)
rfm = model_base.groupby('Customer ID', as_index=False).agg(
    last_order=('Order Date', 'max'),
    frequency=('Order ID', 'nunique'),
    monetary=('Sales', 'sum'),
    valid_lines=('Sales', 'size'),
)
rfm['recency_days'] = (reference_date - rfm['last_order']).dt.days
rfm = rfm[['Customer ID', 'recency_days', 'frequency', 'monetary', 'valid_lines', 'last_order']]
rfm = rfm[(rfm['recency_days'] >= 0) & (rfm['frequency'] > 0) & (rfm['monetary'] > 0)].reset_index(drop=True)
if len(rfm) < 3:
    raise ValueError('Clustering memerlukan sedikitnya tiga pelanggan dengan fitur RFM valid.')
print(f'Baris layak RFM: {len(model_base):,} dari {len(cleaned):,}; pelanggan layak clustering: {len(rfm):,}')
display(rfm.head())

rfm_coverage = 100 * len(model_base) / len(cleaned) if len(cleaned) else 0
print(f'DESKRIPSI OTOMATIS: {rfm_coverage:.1f}% baris transaksi tervalidasi memenuhi syarat RFM; {len(cleaned) - len(model_base):,} baris tidak masuk perhitungan pelanggan.')
print(f'Di antara {len(rfm):,} pelanggan, recency berkisar {rfm["recency_days"].min():,.0f}–{rfm["recency_days"].max():,.0f} hari, frequency {rfm["frequency"].min():,.0f}–{rfm["frequency"].max():,.0f} pesanan, dan median monetary teramati {rfm["monetary"].median():,.2f}.')
print('Lima baris tabel adalah contoh pelanggan; agregasi dan rentang di atas memakai semua pelanggan yang layak.')


## 4.6 Pilih jumlah cluster dan latih K-Means

Script membatasi salinan fitur pada persentil 99, menerapkan `log1p` dan standardisasi, lalu membandingkan K-Means untuk beberapa nilai `k`. Tabel dan grafik silhouette ditafsirkan otomatis; jika pemisahan tidak tersedia, script menerangkan penyebabnya.


In [ ]:
feature_columns = ['recency_days', 'frequency', 'monetary']
rfm_original = rfm[feature_columns].astype(float)
cap_99 = rfm_original.quantile(0.99)
rfm_for_model = rfm_original.clip(upper=cap_99, axis=1)
scaled_features = StandardScaler().fit_transform(np.log1p(rfm_for_model))

score_rows = []
for k in range(2, min(8, len(rfm) - 1) + 1):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(scaled_features)
    if 1 < len(np.unique(labels)) < len(rfm):
        score_rows.append({'k': k, 'silhouette': silhouette_score(scaled_features, labels)})
silhouette_table = pd.DataFrame(score_rows)
if silhouette_table.empty:
    best_k = 1
    rfm['cluster'] = 0
    print('Fitur terlalu seragam untuk pemisahan yang stabil; seluruh pelanggan ditempatkan di cluster 0.')
else:
    best_k = int(silhouette_table.loc[silhouette_table['silhouette'].idxmax(), 'k'])
    rfm['cluster'] = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(scaled_features)
    display(silhouette_table)
    print(f'Jumlah cluster terpilih: {best_k}; skor silhouette: {silhouette_table["silhouette"].max():.3f}')
if not silhouette_table.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(silhouette_table['k'], silhouette_table['silhouette'], marker='o')
    ax.axvline(best_k, color='tab:red', linestyle='--', label=f'k terpilih = {best_k}')
    ax.set_xticks(silhouette_table['k'])
    ax.set_xlabel('Jumlah cluster (k)')
    ax.set_ylabel('Rata-rata silhouette')
    ax.set_title('Perbandingan kualitas pemisahan cluster')
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

if silhouette_table.empty:
    print('DESKRIPSI OTOMATIS: tabel dan grafik silhouette tidak tersedia karena tidak ada kandidat k yang membentuk dua atau lebih kelompok valid.')
else:
    ranked = silhouette_table.sort_values('silhouette', ascending=False)
    print(f'DESKRIPSI OTOMATIS: {len(silhouette_table)} kandidat k dibandingkan. Skor tertinggi {ranked.iloc[0]["silhouette"]:.3f} terjadi pada k={int(ranked.iloc[0]["k"])}; grafik menandainya dengan garis merah.')
    print(f'Selisih skor terbaik dan terendah adalah {ranked.iloc[0]["silhouette"] - ranked.iloc[-1]["silhouette"]:.3f}. Skor ini hanya menggambarkan pemisahan internal pada data saat ini.')


## 4.7 Ringkasan dan visualisasi cluster

Script membuat tabel jumlah pelanggan dan median RFM per cluster, grafik sebar frequency–monetary, serta diagram batang ukuran dan monetary cluster. Uraian otomatis membandingkan cluster berdasarkan median dan ukuran sehingga grafik dapat dibaca tanpa menebak maknanya.


In [ ]:
cluster_summary = rfm.groupby('cluster', as_index=False).agg(
    jumlah_pelanggan=('Customer ID', 'size'),
    median_recency_hari=('recency_days', 'median'),
    median_frequency=('frequency', 'median'),
    median_monetary=('monetary', 'median'),
    total_monetary_teramati=('monetary', 'sum'),
).sort_values('cluster')
display(cluster_summary)

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(rfm['frequency'], rfm['monetary'], c=rfm['cluster'], cmap='tab10', s=25, alpha=0.7)
ax.set_yscale('log')
ax.set_xlabel('Frequency (jumlah pesanan unik)')
ax.set_ylabel('Monetary teramati (skala log)')
ax.set_title('Segmentasi pelanggan Superstore')
fig.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cluster_summary.plot.bar(x='cluster', y='jumlah_pelanggan', ax=axes[0], legend=False, color='tab:blue')
axes[0].set_title('Jumlah pelanggan per cluster')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Pelanggan')
cluster_summary.plot.bar(x='cluster', y='median_monetary', ax=axes[1], legend=False, color='tab:green')
axes[1].set_title('Median monetary teramati per cluster')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Sales')
plt.tight_layout()
plt.show()

largest_cluster = cluster_summary.loc[cluster_summary['jumlah_pelanggan'].idxmax()]
highest_monetary = cluster_summary.loc[cluster_summary['median_monetary'].idxmax()]
lowest_recency = cluster_summary.loc[cluster_summary['median_recency_hari'].idxmin()]
print(f'DESKRIPSI OTOMATIS: cluster terbesar adalah {int(largest_cluster["cluster"])} dengan {int(largest_cluster["jumlah_pelanggan"]):,} pelanggan ({100 * largest_cluster["jumlah_pelanggan"] / len(rfm):.1f}% dari seluruh pelanggan).')
print(f'Median monetary teramati tertinggi terdapat pada cluster {int(highest_monetary["cluster"])} ({highest_monetary["median_monetary"]:,.2f}); pembelian paling baru menurut median recency terdapat pada cluster {int(lowest_recency["cluster"])} ({lowest_recency["median_recency_hari"]:,.0f} hari).')
print('Titik pada grafik sebar mewakili pelanggan; diagram batang kiri membandingkan jumlah pelanggan dan diagram kanan membandingkan median monetary. Grafik sebar memakai skala log pada monetary.')


## 4.8 Audit Akhir dan Pembahasan Otomatis

Script merangkum perubahan kualitas data, proporsi transaksi yang memenuhi syarat RFM, pilihan jumlah cluster, serta profil tiap kelompok. Narasi dihasilkan dari hasil aktual cell sebelumnya sehingga menjawab rumusan masalah pada BAB 1.

Script menyatukan audit akhir dalam tabel, membandingkan sel kosong sebelum dan sesudah per kolom, lalu menampilkan grafiknya. Interpretasi otomatis menjawab pertanyaan kualitas data, cakupan RFM, hasil clustering, dan perubahan terbesar pada nilai kosong.


In [ ]:
audit = {
    'baris_sumber': initial_rows,
    'sel_kosong_sumber': initial_missing,
    'duplikat_persis_dihapus': duplicate_removed,
    'customer_id_dipulihkan': customer_id_from_name + customer_id_from_order,
    'order_date_dipulihkan': order_date_from_order,
    'sales_tidak_valid': invalid_sales,
    'quantity_tidak_valid': invalid_quantity,
    'discount_tidak_valid': invalid_discount,
    'ship_date_dikoreksi': ship_before_order,
    'baris_transaksi_tervalidasi': len(cleaned),
    'sel_kosong_tersisa': int(cleaned.isna().sum().sum()),
    'baris_layak_rfm': len(model_base),
    'pelanggan_di_cluster': len(rfm),
    'jumlah_cluster': best_k,
}
audit_table = pd.DataFrame([{'metrik': key, 'nilai': value} for key, value in audit.items()])
missing_after = cleaned.isna().sum().astype(int)
missing_comparison = pd.DataFrame({
    'kolom': cleaned.columns,
    'kosong_sebelum': initial_missing_by_column.reindex(cleaned.columns, fill_value=0).to_numpy(),
    'kosong_sesudah': missing_after.reindex(cleaned.columns, fill_value=0).to_numpy(),
})
display(audit_table)
display(missing_comparison)

retention_pct = 100 * len(model_base) / len(cleaned) if len(cleaned) else 0
missing_reduction_pct = 100 * (initial_missing - audit['sel_kosong_tersisa']) / initial_missing if initial_missing else 0
print('PEMBAHASAN OTOMATIS BAB 4')
print(f'1. Kualitas data: {initial_rows:,} baris sumber memiliki {initial_missing:,} sel kosong dan {initial_duplicates:,} duplikat persis.')
print(f'   Setelah pembersihan, {duplicate_removed:,} duplikat dihapus; sel kosong tersisa {audit["sel_kosong_tersisa"]:,} ({missing_reduction_pct:.1f}% lebih sedikit dari sumber).')
print(f'   Pemulihan berbasis pasangan unik mengisi {audit["customer_id_dipulihkan"]:,} Customer ID dan {order_date_from_order:,} Order Date.')
print(f'2. Kelayakan RFM: {len(model_base):,} dari {len(cleaned):,} baris transaksi tervalidasi ({retention_pct:.1f}%) memiliki kunci dan Sales valid.')
print(f'   Terdapat {len(rfm):,} pelanggan yang dapat dikelompokkan.')
if silhouette_table.empty:
    print('3. Evaluasi cluster: data tidak menghasilkan lebih dari satu cluster yang valid; skor silhouette tidak tersedia.')
else:
    best_score = float(silhouette_table['silhouette'].max())
    print(f'3. Evaluasi cluster: k={best_k} terpilih dengan skor silhouette {best_score:.3f} dari kandidat yang diuji.')

med_recency = float(rfm['recency_days'].median())
med_frequency = float(rfm['frequency'].median())
med_monetary = float(rfm['monetary'].median())
print('4. Profil kelompok dibandingkan median seluruh pelanggan:')
for row in cluster_summary.itertuples(index=False):
    r = 'lebih baru' if row.median_recency_hari < med_recency else 'lebih lama atau sama'
    f = 'lebih sering' if row.median_frequency > med_frequency else 'lebih jarang atau sama'
    m = 'lebih tinggi' if row.median_monetary > med_monetary else 'lebih rendah atau sama'
    print(f'   Cluster {row.cluster}: {row.jumlah_pelanggan:,} pelanggan; pembelian {r}, frekuensi {f}, monetary {m}.')
print('Interpretasi cluster bersifat deskriptif pada transaksi yang teramati; tidak menyatakan sebab-akibat atau nilai masa depan.')
missing_plot = missing_comparison.loc[
    (missing_comparison['kosong_sebelum'] > 0) | (missing_comparison['kosong_sesudah'] > 0)
].set_index('kolom')[['kosong_sebelum', 'kosong_sesudah']]
if not missing_plot.empty:
    ax = missing_plot.plot.bar(figsize=(max(9, len(missing_plot) * 0.7), 4.5))
    ax.set_title('Nilai kosong per kolom sebelum dan sesudah preprocessing')
    ax.set_xlabel('Kolom')
    ax.set_ylabel('Jumlah sel kosong')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

changed_missing = missing_comparison.assign(perubahan=lambda frame: frame['kosong_sesudah'] - frame['kosong_sebelum'])
most_reduced = changed_missing.loc[changed_missing['perubahan'].idxmin()]
most_increased = changed_missing.loc[changed_missing['perubahan'].idxmax()]
print(f'DESKRIPSI OTOMATIS TABEL/GRAFIK NILAI KOSONG: penurunan terbesar terjadi pada {most_reduced["kolom"]} ({-int(most_reduced["perubahan"]):,} sel lebih sedikit).')
if most_increased['perubahan'] > 0:
    print(f'Kenaikan terbesar terjadi pada {most_increased["kolom"]} ({int(most_increased["perubahan"]):,} sel lebih banyak), kemungkinan akibat validasi nilai yang tidak sah.')
else:
    print('Tidak ada kolom yang jumlah nilai kosongnya meningkat setelah preprocessing.')
print('Grafik hanya memuat kolom yang mempunyai nilai kosong sebelum atau sesudah pembersihan; tabel memuat semua kolom.')


## 4.9 Analisis otomatis untuk kesimpulan, keterbatasan, dan saran

Sel ini menyusun pernyataan hasil dari data yang baru diproses. Gunakan angka di bawah ini bersama tabel dan grafik BAB 4 saat menulis atau meninjau simpulan penelitian.

Script menyusun kesimpulan, keterbatasan, dan saran langsung dari angka yang dihitung pada bagian sebelumnya. Teks yang dicetak adalah interpretasi otomatis hasil penelitian untuk workbook yang diunggah.


In [ ]:
print("DESKRIPSI OTOMATIS: bagian ini menafsirkan hasil audit, segmentasi, keterbatasan, dan tindakan lanjut sesuai angka yang dihitung pada workbook saat ini.")
print('KESIMPULAN OTOMATIS')
print(f'1. Audit menemukan {initial_missing:,} sel kosong dan {initial_duplicates:,} baris duplikat persis pada {initial_rows:,} baris sumber.')
print(f'2. Preprocessing menghapus {duplicate_removed:,} duplikat, memulihkan {audit["customer_id_dipulihkan"]:,} Customer ID dan {order_date_from_order:,} Order Date dari pasangan unik.')
print(f'3. Sebanyak {len(model_base):,} baris layak RFM menghasilkan {len(rfm):,} pelanggan untuk analisis clustering.')
if silhouette_table.empty:
    print('4. Data ini belum mendukung pemisahan menjadi beberapa cluster yang valid.')
else:
    print(f'4. K-Means memilih {best_k} cluster dengan skor silhouette terbaik {best_score:.3f}; karakter tiap cluster ditunjukkan oleh median RFM pada tabel BAB 4.')
print('KETERBATASAN OTOMATIS')
print(f'1. Sebanyak {len(cleaned) - len(model_base):,} baris transaksi tidak memiliki semua kunci atau Sales valid untuk RFM.')
print(f'2. Data transaksi masih memiliki {audit["sel_kosong_tersisa"]:,} sel kosong yang tidak dapat dipulihkan dengan aturan yang digunakan.')
print('3. Monetary adalah jumlah Sales teramati; nilai yang hilang dapat membuat total pelanggan lebih rendah daripada total sebenarnya.')
print('4. Silhouette hanya menilai pemisahan internal pada dataset ini; stabilitas lintas periode dan kegunaan bisnis belum diuji.')
print('5. Pembatasan fitur pada persentil 99 dapat memengaruhi jarak antar pelanggan, walaupun nilai RFM asli tetap disimpan.')
print('SARAN OTOMATIS')
if retention_pct < 80:
    print('1. Periksa pola nilai hilang pada kunci dan Sales sebelum memakai segmentasi untuk keputusan bisnis, karena kurang dari 80% baris layak RFM.')
else:
    print('1. Periksa sampel nilai hilang dan transaksi ekstrem terhadap dokumen sumber untuk menilai kelayakan hasil.')
if not silhouette_table.empty and best_score < 0.5:
    print('2. Bandingkan fitur RFM, transformasi, dan metode clustering lain karena pemisahan internal masih terbatas.')
else:
    print('2. Uji kestabilan cluster pada periode transaksi lain atau sampel pelanggan yang berbeda.')
print('3. Gunakan profil median RFM untuk memberi nama bisnis pada cluster setelah interpretasi bersama pemilik data.')
print('4. Bandingkan hasil dengan strategi penanganan nilai hilang lain hanya jika asumsi imputasi dapat dipertanggungjawabkan.')

## 4.10 Simpan hasil dan unduh ZIP

ZIP berisi transaksi tervalidasi, fitur pelanggan dan label cluster, ringkasan cluster, audit proses, perbandingan nilai kosong, serta tabel silhouette.

Script menyimpan enam tabel hasil sebagai CSV dan mengemasnya dalam ZIP untuk diunduh. Keluaran otomatis mencantumkan nama serta jumlah berkas agar isi arsip dapat diperiksa.


In [ ]:
output_dir = Path('hasil_preprocessing_superstore')
output_dir.mkdir(exist_ok=True)
outputs = {
    'transaksi_tervalidasi.csv': cleaned,
    'pelanggan_rfm_cluster.csv': rfm,
    'ringkasan_cluster.csv': cluster_summary,
    'audit_preprocessing.csv': audit_table,
    'perbandingan_missing.csv': missing_comparison,
    'laporan_outlier_iqr.csv': outlier_report,
}
for filename, frame in outputs.items():
    frame.to_csv(output_dir / filename, index=False, date_format='%Y-%m-%d')
archive_path = Path('hasil_preprocessing_superstore.zip')
with ZipFile(archive_path, 'w', ZIP_DEFLATED) as archive:
    for filename in outputs:
        archive.write(output_dir / filename, arcname=filename)
print(f'Hasil disimpan: {archive_path} ({len(outputs)} file CSV)')
files.download(str(archive_path))

print('DESKRIPSI OTOMATIS ARSIP: ' + ', '.join(f'{name} ({len(frame):,} baris)' for name, frame in outputs.items()))
print('CSV menyimpan tabel hasil yang ditampilkan dalam BAB 4; arsip ZIP tersedia melalui unduhan browser Colab.')


# BAB 5 — KESIMPULAN, KETERBATASAN, DAN SARAN

Bab ini merangkum jawaban penelitian. Angka spesifik, profil cluster, dan rekomendasi yang mengikuti kondisi data dihasilkan otomatis pada **Bagian 4.8–4.9** ketika workbook diunggah dan notebook dijalankan.

## 5.1 Kesimpulan

1. Audit dan preprocessing menghasilkan ukuran yang dapat ditelusuri untuk nilai hilang, duplikat, pemulihan identitas/tanggal, serta proporsi baris yang layak membentuk RFM. Tabel perbandingan nilai kosong pada Bagian 4.8 menunjukkan perubahan per kolom.
2. RFM mengubah transaksi yang memenuhi syarat menjadi ringkasan perilaku pelanggan. Tabel profil cluster dan grafik pada Bagian 4.7 menunjukkan perbedaan recency, frequency, dan monetary **teramati**; hasil aktualnya harus dibaca dari output eksekusi.
3. Jumlah cluster dipilih menggunakan silhouette dari kandidat yang dapat diuji. Grafik pada Bagian 4.6 memperlihatkan perbandingan skor, sedangkan keluaran otomatis Bagian 4.9 menyebut nilai terbaik atau menyatakan jika pemisahan tidak valid. Kesimpulan ini bersifat deskriptif untuk dataset yang diunggah.

## 5.2 Keterbatasan

Baris dengan kunci atau `Sales` tidak valid tidak masuk analisis RFM sehingga profil pelanggan dapat dipengaruhi pola data hilang. Monetary hanya menjumlahkan nilai yang teramati. Pembatasan fitur persentil 99, transformasi, dan pilihan jarak K-Means dapat memengaruhi batas kelompok. Silhouette adalah validasi internal; notebook belum menguji kestabilan antarperiode, representativitas pelanggan, atau manfaat bisnis. Jumlah aktual baris yang tidak masuk RFM dan sel kosong tersisa dicetak otomatis pada Bagian 4.9.

## 5.3 Saran

Periksa sampel nilai hilang dan transaksi ekstrem terhadap dokumen sumber. Jika cakupan RFM rendah, dahulukan perbaikan pengumpulan data dan evaluasi mekanisme nilai hilang. Bandingkan hasil dengan alternatif fitur dan metode clustering; uji kestabilan pada periode atau sampel lain. Nama dan tindakan bisnis untuk setiap cluster perlu divalidasi bersama pemilik data. Saran otomatis pada Bagian 4.9 menyesuaikan kondisi hasil ketika notebook dijalankan.

# REFERENSI

[1] P. S. Fader, B. G. S. Hardie, dan K. L. Lee, “RFM and CLV: Using Iso-Value Curves for Customer Base Analysis,” *Journal of Marketing Research*, vol. 42, no. 4, hlm. 415–430, 2005. [doi:10.1509/jmkr.2005.42.4.415](https://doi.org/10.1509/jmkr.2005.42.4.415).

[2] P. Anitha dan M. M. Patil, “RFM model for customer purchase behavior using K-Means algorithm,” *Journal of King Saud University – Computer and Information Sciences*, vol. 34, no. 5, hlm. 1785–1792, 2022. [doi:10.1016/j.jksuci.2019.12.011](https://doi.org/10.1016/j.jksuci.2019.12.011).

[3] S. P. Lloyd, “Least squares quantization in PCM,” *IEEE Transactions on Information Theory*, vol. 28, no. 2, 1982. [doi:10.1109/TIT.1982.1056489](https://doi.org/10.1109/TIT.1982.1056489).

[4] P. J. Rousseeuw, “Silhouettes: A graphical aid to the interpretation and validation of cluster analysis,” *Journal of Computational and Applied Mathematics*, vol. 20, hlm. 53–65, 1987. [doi:10.1016/0377-0427(87)90125-7](https://doi.org/10.1016/0377-0427(87)90125-7).

[5] D. J. Stekhoven dan P. Bühlmann, “MissForest—non-parametric missing value imputation for mixed-type data,” *Bioinformatics*, vol. 28, no. 1, hlm. 112–118, 2012. [doi:10.1093/bioinformatics/btr597](https://doi.org/10.1093/bioinformatics/btr597).

[6] D. A. Awaliyah, B. Prasetiyo, R. Muzayanah, dan A. D. Lestari, “Optimizing Customer Segmentation in Online Retail Transactions through the Implementation of the K-Means Clustering Algorithm,” *Scientific Journal of Informatics*, vol. 11, no. 2, hlm. 539–548, 2024. [doi:10.15294/sji.v11i2.6137](https://doi.org/10.15294/sji.v11i2.6137).

[7] R. Y. Wang dan D. M. Strong, “Beyond Accuracy: What Data Quality Means to Data Consumers,” *Journal of Management Information Systems*, vol. 12, no. 4, hlm. 5–33, 1996. [doi:10.1080/07421222.1996.11518099](https://doi.org/10.1080/07421222.1996.11518099).

[8] R. Xu dan D. Wunsch II, “Survey of clustering algorithms,” *IEEE Transactions on Neural Networks*, vol. 16, no. 3, hlm. 645–678, 2005. [doi:10.1109/TNN.2005.845141](https://doi.org/10.1109/TNN.2005.845141).

[9] J. A. C. Sterne dkk., “Multiple imputation for missing data in epidemiological and clinical research: potential and pitfalls,” *BMJ*, vol. 338, b2393, 2009. [doi:10.1136/bmj.b2393](https://doi.org/10.1136/bmj.b2393).

[10] M. Hubert dan E. Vandervieren, “An adjusted boxplot for skewed distributions,” *Computational Statistics & Data Analysis*, vol. 52, no. 12, hlm. 5186–5201, 2008. [doi:10.1016/j.csda.2007.11.008](https://doi.org/10.1016/j.csda.2007.11.008).

### Dokumentasi implementasi

- [Google Colab: upload dan download berkas](https://colab.research.google.com/notebooks/io.ipynb).
- [pandas: `read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html).
- [scikit-learn: KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) dan [silhouette score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).